In [ ]:
import sys
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pre_tagger import PreTagger

print(">> Moduli caricati")

In [ ]:
# --- Tag candidati da assegnare durante il pre-tagging ---
# Modifica questa lista con i tag che vuoi far valutare da TagAssigner
candidate_tags = [
    "tag_1",
    "tag_2",
    "tag_3",
]

# Path del file sidecar su cui verranno salvati i tag assegnati
sidecar_path = project_root / "data/processed/test/sidecar_04_05T.json"

# Dimensione della pagina di scroll da Qdrant (controlla anche ogni quanti
# chunk viene fatta una scrittura sul sidecar, vedi PreTagger.run)
BATCH_SIZE = 100

print(f"?> Tag candidati: {candidate_tags}")
print(f"?> Sidecar: {sidecar_path}")

In [ ]:
pretagger = PreTagger(sidecar_path=str(sidecar_path))

try:
    pretagger.run(
        candidate_tags=candidate_tags,
        batch_size=BATCH_SIZE,
    )
finally:
    # Rilascia evenutali risorse che il pretagger ha autonomamente istanziato
    pretagger.close()

In [ ]:
from src.sidecar_manager import SidecarManager

# Istanza di sola lettura del sidecar per ispezionare il risultato:
#  poiché chiusa quella usata dal pre-tagger, non darà errori
verifier = SidecarManager(filepath=str(SIDECAR_PATH))
global_tags = verifier.get_global_tags()

# Prova globale
print(f"!>> Tag globali registrati: {len(global_tags)}")
for tag, info in global_tags.items():
    print(f"  - {tag}: conteggio={info.get('count')}, colore={info.get('color')}")

# Campione dei chunk taggati
sidecar_data = verifier.load_data()
overrides = sidecar_data.get("tag_overrides", {})
print(f"\n!>> Totale chunk con tag assegnati: {len(overrides)}")

if overrides:
    print("Sample primi 5 chunk:")
    for chunk_id, info in list(overrides.items())[:5]:
        print(f"  - [{chunk_id}] -> {info.get('user_tags')}")